# SILVER LAYER SCRIPT

### Data Acess Using Application


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.conf.set("fs.azure.account.auth.type.azurdatalakestorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.azurdatalakestorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.azurdatalakestorage.dfs.core.windows.net", <ADD_ID_HERE>)
spark.conf.set("fs.azure.account.oauth2.client.secret.azurdatalakestorage.dfs.core.windows.net", <ADD_SECRET_HERE>)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.azurdatalakestorage.dfs.core.windows.net", <ADD_ID_HERE>)

In [0]:
df_cal = spark.read.format('csv')\
              .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Calendar")

In [0]:
df_cus = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Customers")

In [0]:
df_cat = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Product_Categories")

In [0]:
df_subc = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Product_Subcategories")

In [0]:
df_prod = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Products")

In [0]:
df_ret = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Returns")

In [0]:
df_sales = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Sales*")

In [0]:
df_terr = spark.read.format('csv')\
                .option("header", "true").option("inferSchema", "true").load("abfss://bronze@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Territories")

## Transformation

### calender

In [0]:
df_cal.display()

In [0]:
df_cal = df_cal.withColumn('Month', month(col('Date')))\
               .withColumn('Year', year(col('Date')))
               



In [0]:
df_cal.write.format("parquet").mode("append").option('path', 'abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Calendar').save()

### customers

In [0]:
df_cus.display()

In [0]:
df_cus.withColumn('FullName' ,concat(col("Prefix") , lit(' ') , col("FirstName") ,lit(' ') , col( "LastName")))

In [0]:
df_cus = df_cus.withColumn('Fullname' , concat_ws(' ' , col('prefix') , col('firstname') , col('lastname')))

In [0]:
df_cus.display()

In [0]:
df_cus.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Customers") \
    .save()

### product subcategories

In [0]:
df_subc.display()

In [0]:
df_subc.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Product_Subcategories") \
    .save()

### Products

In [0]:
    df_prod.display()

In [0]:
df_prod = df_prod.withColumn('ProductSKU' , split(col('ProductSKU') , '-')[0]) \
                 .withColumn('ProductName' , split(col('ProductName') , ' ')[0]) 
                 
               
 

In [0]:
df_prod.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Products") \
    .save()

### Returns

In [0]:
df_ret.display()

In [0]:
df_ret.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_returns") \
    .save()

### Territories

In [0]:
df_terr.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_Terriorties") \
    .save()

### Sales

In [0]:
df_sales.display()

In [0]:
df_sales = df_sales.withColumn('StockDate' ,to_timestamp('StockDate')) 

                 

In [0]:
df_sales = df_sales.withColumn('OrderNumber' , regexp_replace(col('OrderNumber') , 'S' ,'T')) 
              

In [0]:
df_sales = df_sales.withColumn('Multiply' , col('orderlineitem') * col('orderquantity')) 
    

In [0]:
df_sales.write.format("parquet") \
    .mode("append") \
    .option("path", "abfss://silver@azurdatalakestorage.dfs.core.windows.net/AdventureWorks_sales") \
    .save()

### Charts

In [0]:
df_terr.display()